In [2]:
# Competition-Solution/notebooks/train/01_data_preprocessing/00_data_preprocessing.ipynb

### Data Preprocessing

The notebooks within the **`01_data_preprocessing`** folder detail the specific preprocessing steps applied to the trial dataset for each of the three subtasks:

-   **Subtask 1:** Call2Action
-   **Subtask 2:** Attacks on the Democratic Basic Order (FDGO)
-   **Subtask 3:** Violence Detection

Data preprocessing involves the cleaning and transforming of the raw text data into a suitable format. The preprocessing stage draws from the insights gained from the prior Data Profiling phase (documented in `00_data_profiling`). (e.g. tweet lengths, class imbalance, duplicate tweet descriptions etc.)

The primary goals of the preprocessing steps outlined here are to:
*   Remove or replace noise and irrelevant elements commonly found in social media text that are not beneficial for transformer based models (e.g., URLs, special characters, excessive whitespace).
*   Handle dataset-specific artifacts (e.g., anonymization tokens like `[@GRP]`).
*   Create a dataset split.
*   Handle overlap between the training and validation set.
*   Saving the datasets to disk, so that we can use them for the subsequent model training.

The specific preprocessing decisions and their implementation are guided by the characteristics identified during the data profiling stage. A consistent and well-reasoned preprocessing pipeline ensures that the models learn from meaningful signals instead of noise. The cleaned data resulting from this stage will form the input for subsequent tokenization and model training.

---

##### <b>Imports</b>

In [3]:
import sys
from rich.console import Console

# HuggingFace
from datasets import DatasetInfo, Features, Value, ClassLabel, Dataset, DatasetDict

sys.path.append("../../../src")
import data_utils

console = Console()

##### <b>Loading the data</b>

In [4]:
# Loads the competition training data
df_call2action, df_dbo, df_violence = data_utils.load_competition_data(data_dir="../../../data/raw/", type="train")

# Loads the competition hidden test-set data
df_call2action_test, df_dbo_test, df_violence_test = data_utils.load_competition_data(data_dir="../../../data/raw/", type="test")

Loading competition data...

Data of type 'train' loaded successfully.

Loading competition data...

Data of type 'test' loaded successfully.

##### Checking URL and email occurances

- C2A

In [5]:
c2a_url_count = data_utils.check_for_urls(df_call2action, column_name="description", show_random_hit_examples=True)
c2a_email_count = data_utils.check_for_emails(df_call2action, column_name="description", show_random_hit_examples=True)

Random examples containing URLs:

- 
http://www.focus.de/wissen/mensch/geschichte/serie-fuenf-irrtuemer-ueber-die-nachkriegszeit-mythos-truemmerfrauen-d
en-schutt-raeumten-in-wirklichkeit-andere-weg_id_4681101.html

- http://youtu.be/FQXlUF4hN2E

- https://www.youtube.com/watch?v=B4Wf2ErF71I

Total URLs found in the 'description' column of dataframe 'Call2Action': 104

No emails found in the 'description' column of dataframe 'Call2Action.'

##### Checking URL and email occurances *(DBO)*

- DBO

In [6]:
dbo_url_count = data_utils.check_for_urls(df_dbo, column_name="description", show_random_hit_examples=True)
dbo_email_count = data_utils.check_for_emails(df_dbo, column_name="description", show_random_hit_examples=True)

Random examples containing URLs:

- http://antiakhondha.blogspot.com

- https://twitter.com/IamTeamIK/status/666481561131483137

- http://img2.faszination-fankurve.de/images/news/1442102789.jpg

Total URLs found in the 'description' column of dataframe 'DBO': 208

No emails found in the 'description' column of dataframe 'DBO.'

##### Checking URL and email occurances *(Violence)*

- VIO

In [7]:
vio_url_count = data_utils.check_for_urls(df_violence, column_name="description", show_random_hit_examples=True)
vio_email_count = data_utils.check_for_emails(df_violence, column_name="description", show_random_hit_examples=True)

Random examples containing URLs:

- 
http://www.abendblatt.de/hamburg/hamburg-mitte/article205440131/Ehemalige-Kapernaum-Kirche-jetzt-mit-Allah-Schriftz
ug.html#

- http://www.volksstimme.de/nachrichten/magdeburg/1439109_Bis-Ende-des-Jahres-rund-980-Unterkuenfte-noetig.html

- 
https://m.facebook.com/alternativefuerde/photos/a.542889462408064.1073741828.540404695989874/1032926653404340/?type
=3

Total URLs found in the 'description' column of dataframe 'Violence': 212

No emails found in the 'description' column of dataframe 'Violence.'

##### <b>Cleaning the text</b>

In [8]:
# Replaces the URLs and emails with placeholder tokens
if df_call2action is not None and df_dbo is not None and df_violence is not None:
    df_call2action_cleaned = data_utils.clean_text(df_call2action, column_name="description", url_replacement="[@URL]", email_replacement="[@EMAIL]")
    df_dbo_cleaned = data_utils.clean_text(df_dbo, column_name="description", url_replacement="[@URL]", email_replacement="[@EMAIL]")
    df_violence_cleaned = data_utils.clean_text(df_violence, column_name="description", url_replacement="[@URL]", email_replacement="[@EMAIL]")
else:
    console.print("DataFrames not loaded. Please check the data loading path in the previous cell.", style="bold red")

<p>Verifying the data cleaning process</p>

- C2A

In [9]:
c2a_url_count_cleaned = data_utils.check_for_urls(df_call2action_cleaned, column_name="description", show_random_hit_examples=True)
c2a_email_count_cleaned = data_utils.check_for_emails(df_call2action_cleaned, column_name="description", show_random_hit_examples=True)

No URLs found in the 'description' column of dataframe 'unnamed.

No emails found in the 'description' column of dataframe 'unnamed.'

- DBO

In [10]:
dbo_url_count_cleaned = data_utils.check_for_urls(df_dbo_cleaned, column_name="description", show_random_hit_examples=True)
dbo_email_count_cleaned = data_utils.check_for_emails(df_dbo_cleaned, column_name="description", show_random_hit_examples=True)

No URLs found in the 'description' column of dataframe 'unnamed.

No emails found in the 'description' column of dataframe 'unnamed.'

- VIO

In [11]:
violence_url_count_cleaned = data_utils.check_for_urls(df_violence_cleaned, column_name="description", show_random_hit_examples=True)
violence_email_count_cleaned = data_utils.check_for_emails(df_violence_cleaned, column_name="description", show_random_hit_examples=True)

No URLs found in the 'description' column of dataframe 'unnamed.

No emails found in the 'description' column of dataframe 'unnamed.'

### Saving the cleaned dataframes to csv files for further use

In [12]:
data_utils.save_dataframes_to_csv(
    dataframes_dict={
        "c2a_cleaned": (df_call2action_cleaned, "c2a"),
        "dbo_cleaned": (df_dbo_cleaned, "dbo"),
        "vio_cleaned": (df_violence_cleaned, "vio")
    },
    base_directory="../../../data/processed/",
    suffix="train"
)

Saving dataframe 'c2a_cleaned' to ../../../data/processed/c2a/c2a_cleaned_train.csv...

File 'c2a_cleaned_train.csv' already exists. Overwriting...

Dataframe 'c2a_cleaned' saved successfully.

Saving dataframe 'dbo_cleaned' to ../../../data/processed/dbo/dbo_cleaned_train.csv...

File 'dbo_cleaned_train.csv' already exists. Overwriting...

Dataframe 'dbo_cleaned' saved successfully.

Saving dataframe 'vio_cleaned' to ../../../data/processed/vio/vio_cleaned_train.csv...

File 'vio_cleaned_train.csv' already exists. Overwriting...

Dataframe 'vio_cleaned' saved successfully.

All dataframes saved successfully.

### Train/Validation Splitting

##### Converting the cleaned pandas dataframes into HuggingFace Datasets

In [13]:
# Training data Dataset Info objects
c2a_info = DatasetInfo(
    description="Training data for binary detection of call-to-actions in German tweets from right-wing extremist network (2014-2016).",
    features=Features({
        "id": Value("string"),
        "description": Value("string"),
        "C2A": ClassLabel(names=["False", "True"])
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition",
    supervised_keys=("description", "C2A")
)

dbo_info = DatasetInfo(
    description="Training Data for Four-class classification of statements against the free democratic basic order in German tweets.",
    features=Features({
        "id": Value("string"),
        "description": Value("string"),
        "DBO": ClassLabel(names=["nothing", "criticism", "agitation", "subversive"])
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition",
    supervised_keys=("description", "DBO")
)

vio_info = DatasetInfo(
    description="Training Data for Binary detection of violence-related statements in German tweets from right-wing extremist network.",
    features=Features({
        "id": Value("string"),
        "description": Value("string"),
        "VIO": ClassLabel(names=["False", "True"])
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition",
    supervised_keys=("description", "VIO")
)

In [14]:
# Converts the cleaned training data pandas dataframes into HuggingFace Datasets
c2a_dataset = Dataset.from_pandas(
    df=df_call2action_cleaned,
    features=c2a_info.features,
    info=c2a_info,
    preserve_index=False
)

dbo_dataset = Dataset.from_pandas(
    df=df_dbo_cleaned,
    features=dbo_info.features,
    info=dbo_info,
    preserve_index=False
)

vio_dataset = Dataset.from_pandas(
    df=df_violence_cleaned,
    features=vio_info.features,
    info=vio_info,
    preserve_index=False
)

In [ ]:
# Test data Dataset Info objects
c2a_test_info = DatasetInfo(
    description="Test Data for binary detection of call-to-actions in German tweets from right-wing extremist network (2014-2016).",
    features=Features({
        "id": Value("string"),
        "description": Value("string")
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition"
)

dbo_test_info = DatasetInfo(
    description="Test data for four-class classification of statements against the free democratic basic order in German tweets.",
    features=Features({
        "id": Value("string"),
        "description": Value("string")
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition"
)

vio_test_info = DatasetInfo(
    description="Test Data for Binary detection of violence-related statements in German tweets from right-wing extremist network.",
    features=Features({
        "id": Value("string"),
        "description": Value("string")
    }),
    homepage="https://www.codabench.org/competitions/4963/",
    citation="GermEval 2025: A Shared Task for the Detection of Harmful Content on Social Media",
    license="Research use only - shared within GermEval 2025 competition"
)

In [16]:
# Converts the test data pandas dataframes into HuggingFace Datasets
c2a_test_dataset = Dataset.from_pandas(
    df=df_call2action_test,
    features=c2a_test_info.features,
    info=c2a_test_info,
    preserve_index=False
)

dbo_test_dataset = Dataset.from_pandas(
    df=df_dbo_test,
    features=dbo_test_info.features,
    info=dbo_test_info,
    preserve_index=False
)

vio_test_dataset = Dataset.from_pandas(
    df=df_violence_test,
    features=vio_test_info.features,
    info=vio_test_info,
    preserve_index=False
)

In [ ]:
# Toggle: use hidden test (unlabeled) or an internal labeled test split
USE_HIDDEN_TEST = True

# No-op assignment to make intent explicit and keep style consistent
if USE_HIDDEN_TEST:
    use_hidden_test = True
else:
    use_hidden_test = False


In [ ]:
# Train/Validation Stratified Split (80/20)
# Note: Shuffles the datasets during splitting to ensure that the data is randomly distributed
# The stratified split ensures that the distribution of the labels in the training and validation sets is the same as in the original dataset

# Preserves the original datasets
c2a_dataset_original = c2a_dataset
dbo_dataset_original = dbo_dataset
vio_dataset_original = vio_dataset

# c2a Splitting
c2a_splits = c2a_dataset.train_test_split(test_size=0.2, seed=42, shuffle=True, stratify_by_column="C2A")
if use_hidden_test:
    c2a_dataset = DatasetDict({
        "train": c2a_splits["train"],
        "validation": c2a_splits["test"],
        "test": c2a_test_dataset
    })
else:
    # Creates an internal test split from the validation split for labeled testing
    internal_c2a = c2a_splits["test"].train_test_split(test_size=0.5, seed=42, shuffle=True, stratify_by_column="C2A")
    c2a_dataset = DatasetDict({
        "train": c2a_splits["train"],
        "validation": internal_c2a["train"],
        "test": internal_c2a["test"]
    })

# dbo Splitting
dbo_splits = dbo_dataset.train_test_split(test_size=0.2, seed=42, shuffle=True, stratify_by_column="DBO")
if use_hidden_test:
    dbo_dataset = DatasetDict({
        "train": dbo_splits["train"],
        "validation": dbo_splits["test"],
        "test": dbo_test_dataset
    })
else:
    internal_dbo = dbo_splits["test"].train_test_split(test_size=0.5, seed=42, shuffle=True, stratify_by_column="DBO")
    dbo_dataset = DatasetDict({
        "train": dbo_splits["train"],
        "validation": internal_dbo["train"],
        "test": internal_dbo["test"]
    })

# vio Splitting
vio_splits = vio_dataset.train_test_split(test_size=0.2, seed=42, shuffle=True, stratify_by_column="VIO")
if use_hidden_test:
    vio_dataset = DatasetDict({
        "train": vio_splits["train"],
        "validation": vio_splits["test"],
        "test": vio_test_dataset
    })
else:
    internal_vio = vio_splits["test"].train_test_split(test_size=0.5, seed=42, shuffle=True, stratify_by_column="VIO")
    vio_dataset = DatasetDict({
        "train": vio_splits["train"],
        "validation": internal_vio["train"],
        "test": internal_vio["test"]
    })

In [18]:
# Verifies the stratification results
console.print("\n[bold yellow]═══ CALL2ACTION STRATIFICATION ═══[/bold yellow]")
data_utils.show_hf_stratification_results(c2a_dataset_original, c2a_dataset, "C2A")

console.print("\n[bold yellow]═══ DBO STRATIFICATION ═══[/bold yellow]")
data_utils.show_hf_stratification_results(dbo_dataset_original, dbo_dataset, "DBO")

console.print("\n[bold yellow]═══ VIOLENCE STRATIFICATION ═══[/bold yellow]")
data_utils.show_hf_stratification_results(vio_dataset_original, vio_dataset, "VIO")

═══ CALL2ACTION STRATIFICATION ═══

C2A Stratification Verification

Original: 6840 | Train: 5472 (80.0%) | Validation: 1368 (20.0%) | Test: 2982 (separate dataset)

┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Class ┃ Original% ┃ Train% ┃ Validation% ┃ Max Diff ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 0     │     90.3% │  90.3% │       90.3% │     0.0% │
│ 1     │      9.7% │   9.7% │        9.7% │     0.0% │
└───────┴───────────┴────────┴─────────────┴──────────┘

═══ DBO STRATIFICATION ═══

...

┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Class ┃ Original% ┃ Train% ┃ Validation% ┃ Max Diff ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 0     │     84.2% │  84.2% │       84.2% │     0.0% │
│ 1     │     10.8% │  10.8% │       10.8% │     0.0% │
│ 2     │      4.2% │   4.2% │        4.2% │     0.0% │
│ 3     │      0.8% │   0.8% │        0.8% │     0.0% │
└───────┴───────────┴────────┴─────────────┴──────────┘

═══ VIOLENCE STRATIFICATION ═══

VIO Stratification Verification

Original: 7783 | Train: 6226 (80.0%) | Validation: 1557 (20.0%) | Test: 3335 (separate dataset)

┏━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Class ┃ Original% ┃ Train% ┃ Validation% ┃ Max Diff ┃
┡━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ 0     │     92.8% │  92.8% │       92.7% │     0.0% │
│ 1     │      7.2% │   7.2% │        7.3% │     0.0% │
└───────┴───────────┴────────┴─────────────┴──────────┘

##### Removing duplicate tweet descriptions

In [19]:
# Removes the duplicate tweet descriptions, that we saw during the data profiling phase, 
# from the validation set as not to leak information to the model, thereby causing a bias in the model's performance.
c2a_dataset = data_utils.handle_dataset_leakage(c2a_dataset)
dbo_dataset = data_utils.handle_dataset_leakage(dbo_dataset)
vio_dataset = data_utils.handle_dataset_leakage(vio_dataset)

Removed 37 duplicate samples from validation set (1368 -> 1331)

Removed 91 duplicate samples from validation set (1491 -> 1400)

Removed 71 duplicate samples from validation set (1557 -> 1486)

In [ ]:
# Saves the training datasets to disk

# Ensures hidden test splits do not contain label columns (safety guard)
if use_hidden_test:
    # c2a
    if "test" in c2a_dataset:
        test_cols = c2a_dataset["test"].column_names
        cols_to_drop = []
        if "C2A" in test_cols:
            cols_to_drop.append("C2A")
        if "labels" in test_cols:
            cols_to_drop.append("labels")
        if len(cols_to_drop) > 0:
            c2a_dataset["test"] = c2a_dataset["test"].remove_columns(cols_to_drop)
    
    # dbo
    if "test" in dbo_dataset:
        test_cols = dbo_dataset["test"].column_names
        cols_to_drop = []
        if "DBO" in test_cols:
            cols_to_drop.append("DBO")
        if "labels" in test_cols:
            cols_to_drop.append("labels")
        if len(cols_to_drop) > 0:
            dbo_dataset["test"] = dbo_dataset["test"].remove_columns(cols_to_drop)
    
    # vio
    if "test" in vio_dataset:
        test_cols = vio_dataset["test"].column_names
        cols_to_drop = []
        if "VIO" in test_cols:
            cols_to_drop.append("VIO")
        if "labels" in test_cols:
            cols_to_drop.append("labels")
        if len(cols_to_drop) > 0:
            vio_dataset["test"] = vio_dataset["test"].remove_columns(cols_to_drop)

c2a_dataset.save_to_disk("../../../data/processed/c2a/c2a_hf_dataset_train")
dbo_dataset.save_to_disk("../../../data/processed/dbo/dbo_hf_dataset_train")
vio_dataset.save_to_disk("../../../data/processed/vio/vio_hf_dataset_train")

Saving the dataset (0/1 shards):   0%|          | 0/5472 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1331 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2982 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5963 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3194 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6226 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1486 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3335 [00:00<?, ? examples/s]

In [19]:
# Viewing the first 5 rows of the training datasets
print(c2a_dataset["train"][:5])
print(dbo_dataset["train"][:5])
print(vio_dataset["train"][:5])

{'id': ['985378574833899', '1111307682240987', '985718668133223', '876360412402383', '882707731767651'], 'description': ['BERLIN ab nach BERLIN !!!!!!!!!!!!!!', 'Eine Freudenbotschaft und das gleich am Wochenanfang. Warum sind eigentlich die Asylhelfer entrüstet.Die können doch ihren Job weiter machen,oder ?', 'Holger Bommel,meine vorherige Antwort ...... na dann beten........ ,, galt auch Fräulein Naumann.  Tolle Antwort Holger  ????????????????????????', 'machen wir schon, sonst wären wir ja gar nicht so viele geworden.', 'Da waren doch bestimmt wieder Pediga - Anhänger am Werk. Wie bringe dem feigen Mord in Dresden.....'], 'C2A': [1, 0, 0, 0, 0]}
{'id': ['1391006127811346432', '1064434893594933', '1405199261382365184', '1584', '991240877581002'], 'description': ['@PeterBe38098838 @reitschuster Und?? War die kleine Anne Frank stolz 🖕', 'Bernd Arnold Zur Mitteilung: Vor 27 Jahren sind wir mit Schildern auf dem Auto „Wählt CDU“ rum gefahren . Jetzt bin ich der Einstigste, der den Mut h